# ARPD: Adaptive Evidence Retrieval with Paraphrase-Robust Training
### Fake News Detection — LIAR Dataset

**Runtime:** GPU T4 (Google Colab free tier)  
**Estimated total time:** ~45-60 phút (bao gồm Wikipedia retrieval)


In [ ]:
# Cell 1: Install dependencies
# Chạy cell này một lần duy nhất sau khi mở Colab

!pip install -q \
    torch>=2.1.0 \
    transformers>=4.38.0 \
    sentence-transformers>=2.6.0 \
    scikit-learn>=1.4.0 \
    numpy>=1.26.0 \
    pandas>=2.2.0 \
    nltk>=3.8.1 \
    wikipedia-api>=0.6.0 \
    datasets>=2.18.0 \
    tqdm>=4.66.0 \
    matplotlib>=3.8.0 \
    seaborn>=0.13.0 \
    tabulate>=0.9.0

# Clone project (thay bằng link repo của bạn, hoặc upload manual)
import os
if not os.path.exists('/content/ARPD-research'):
    !git clone https://github.com/YOUR_USERNAME/ARPD-research.git /content/ARPD-research

os.chdir('/content/ARPD-research')
print('Working directory:', os.getcwd())

In [ ]:
# Cell 2: Imports & Config

import sys
sys.path.insert(0, '/content/ARPD-research')

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path

# Config
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DATA_DIR = Path('/content/ARPD-research/data/processed')
RESULTS_DIR = Path('/content/ARPD-research/results')
DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Seed
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

print(f'Device: {DEVICE}')
print(f'PyTorch: {torch.__version__}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Cell 3: Load LIAR Dataset

from src.data_loader import load_all_splits, save_processed

# Load từ HuggingFace (lần đầu download ~5MB)
print('Loading LIAR dataset...')
splits = load_all_splits()
train_df, val_df, test_df = splits['train'], splits['validation'], splits['test']

# Lưu ra CSV
for name, df in splits.items():
    save_processed(df, DATA_DIR / f'liar_{name}.csv')

# Stats
for name, df in splits.items():
    n_real = df['label'].sum()
    n_fake = len(df) - n_real
    print(f'  {name:12s}: {len(df):5d} total | REAL={n_real} ({n_real/len(df)*100:.1f}%) | FAKE={n_fake} ({n_fake/len(df)*100:.1f}%)')

# Preview
print('\nSample claims:')
train_df[['claim', 'label_str', 'label']].head(5)

In [ ]:
# Cell 4: Baselines
# TF-IDF + LR và DistilBERT (sentence-transformer) + MLP

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sentence_transformers import SentenceTransformer
from src.classifier import ARPDTrainer

baseline_results = []

# --- Baseline 1: TF-IDF + LR ---
print('=== Baseline 1: TF-IDF + Logistic Regression ===')
vec = TfidfVectorizer(max_features=50000, ngram_range=(1, 2), sublinear_tf=True)
X_tr_tfidf = vec.fit_transform(train_df['claim'])
X_val_tfidf = vec.transform(val_df['claim'])
X_te_tfidf = vec.transform(test_df['claim'])

lr = LogisticRegression(max_iter=1000, C=1.0, class_weight='balanced', random_state=SEED)
lr.fit(X_tr_tfidf, train_df['label'])

for split_name, X, y in [('val', X_val_tfidf, val_df['label']), ('test', X_te_tfidf, test_df['label'])]:
    preds = lr.predict(X)
    acc = accuracy_score(y, preds)
    f1 = f1_score(y, preds, average='macro')
    print(f'  {split_name}: acc={acc:.4f}  f1_macro={f1:.4f}')
    if split_name == 'test':
        baseline_results.append({'model': 'TF-IDF+LR', 'test_acc': acc, 'test_f1': f1})

print(classification_report(test_df['label'], lr.predict(X_te_tfidf), target_names=['FAKE','REAL']))

# --- Baseline 2: DistilBERT (all-distilroberta-v1) + MLP ---
print('\n=== Baseline 2: DistilBERT + MLP ===')
db_model = SentenceTransformer('sentence-transformers/all-distilroberta-v1')

def encode_df(df):
    return db_model.encode(df['claim'].tolist(), show_progress_bar=True, batch_size=64)

print('Encoding...')
X_tr_db = encode_df(train_df)
X_val_db = encode_df(val_df)
X_te_db = encode_df(test_df)

db_trainer = ARPDTrainer(input_dim=X_tr_db.shape[1], device=DEVICE)
db_trainer.fit(
    X_tr_db, train_df['label'].values,
    X_val_db, val_df['label'].values,
    epochs=10, patience=3, verbose=True
)

db_metrics = db_trainer.evaluate(X_te_db, test_df['label'].values)
print(f'  test: acc={db_metrics["accuracy"]:.4f}  f1_macro={db_metrics["f1_macro"]:.4f}')
baseline_results.append({'model': 'DistilBERT+MLP', 'test_acc': db_metrics['accuracy'], 'test_f1': db_metrics['f1_macro']})

# Lưu baseline results
df_baseline = pd.DataFrame(baseline_results)
df_baseline.to_csv(RESULTS_DIR / 'baseline_results.csv', index=False)
print('\nBaseline results:')
df_baseline

In [ ]:
# Cell 5: ARPD Full Pipeline
# Chạy full pipeline với adaptive retrieval + synonym augmentation
# CẢNH BÁO: Wikipedia retrieval cho ~10K samples tốn ~2-3 giờ
# Dùng RETRIEVE=False để test nhanh (không retrieval)

from src.pipeline import ARPDPipeline
from sklearn.metrics import classification_report, accuracy_score, f1_score

RETRIEVE = False   # Đổi thành True khi chạy thật
EPOCHS = 20

pipeline = ARPDPipeline(
    k_min=1,
    k_max=5,
    augmentation_method='synonym',
    p_synonym=0.15,
    device=DEVICE,
    retriever_sleep=0.2 if RETRIEVE else 0.0,
)

print('Training ARPD pipeline...')
history = pipeline.fit(
    train_df['claim'].tolist(), train_df['label'].tolist(),
    val_df['claim'].tolist(), val_df['label'].tolist(),
    epochs=EPOCHS,
    retrieve_evidence=RETRIEVE,
    verbose=True,
)

print('\nEvaluating on test set...')
test_preds = pipeline.predict(test_df['claim'].tolist(), retrieve_evidence=RETRIEVE)
y_test = test_df['label'].values

arpd_acc = accuracy_score(y_test, test_preds)
arpd_f1 = f1_score(y_test, test_preds, average='macro')
print(f'\nARPD Test: acc={arpd_acc:.4f}  f1_macro={arpd_f1:.4f}')
print(classification_report(y_test, test_preds, target_names=['FAKE','REAL']))

# Lưu
arpd_result = {'model': 'ARPD', 'test_acc': arpd_acc, 'test_f1': arpd_f1, 'retrieve': RETRIEVE}
pd.DataFrame([arpd_result]).to_csv(RESULTS_DIR / 'arpd_results.csv', index=False)
pd.DataFrame(history).to_csv(RESULTS_DIR / 'arpd_history.csv', index=False)
pipeline.save(RESULTS_DIR / 'checkpoints')
print(f'Saved to {RESULTS_DIR}')

In [ ]:
# Cell 6: Paraphrase Attack Evaluation

from src.paraphrase_augmentor import synonym_substitute, ensure_nltk_data
from sklearn.metrics import accuracy_score, f1_score

ensure_nltk_data()

def make_adversarial(claims, p, seed_offset=0):
    return [synonym_substitute(c, p=p, seed=i+seed_offset) for i, c in enumerate(claims)]

test_claims = test_df['claim'].tolist()
y_test = test_df['label'].values

robustness_rows = []

# Clean
clean_preds = pipeline.predict(test_claims, retrieve_evidence=False)
clean_acc = accuracy_score(y_test, clean_preds)
clean_f1 = f1_score(y_test, clean_preds, average='macro')
robustness_rows.append({'attack': 'clean', 'p_sub': 0.0, 'accuracy': clean_acc, 'f1_macro': clean_f1, 'drop': 0.0})
print(f'Clean:     acc={clean_acc:.4f}  f1={clean_f1:.4f}')

# Adversarial attacks với p ngày càng tăng
for p in [0.10, 0.15, 0.20, 0.30, 0.50]:
    adv_claims = make_adversarial(test_claims, p=p)
    adv_preds = pipeline.predict(adv_claims, retrieve_evidence=False)
    adv_acc = accuracy_score(y_test, adv_preds)
    adv_f1 = f1_score(y_test, adv_preds, average='macro')
    drop = clean_acc - adv_acc
    print(f'Attack p={p:.2f}: acc={adv_acc:.4f}  f1={adv_f1:.4f}  drop={drop:+.4f}')
    robustness_rows.append({'attack': f'synonym_p{int(p*100)}', 'p_sub': p, 'accuracy': adv_acc, 'f1_macro': adv_f1, 'drop': drop})

df_rob = pd.DataFrame(robustness_rows)
df_rob.to_csv(RESULTS_DIR / 'robustness_results.csv', index=False)
print('\nRobustness results:')
df_rob

In [ ]:
# Cell 7: Ablation Study
# So sánh các variants của ARPD:
#   A) No augmentation, no retrieval
#   B) No augmentation, with retrieval (nếu RETRIEVE=True)
#   C) With augmentation, no retrieval   <-- thường chạy ở đây
#   D) Full ARPD (aug + retrieval)

ablation_results = []

configs = [
    {'name': 'No-Aug No-Ret',  'augmentation': 'none',    'retrieve': False},
    {'name': 'Aug No-Ret',     'augmentation': 'synonym', 'retrieve': False},
    # {'name': 'No-Aug With-Ret', 'augmentation': 'none',    'retrieve': True},  # uncomment nếu muốn
    # {'name': 'Full ARPD',      'augmentation': 'synonym', 'retrieve': True},
]

for cfg in configs:
    print(f"\n--- Ablation: {cfg['name']} ---")
    p = ARPDPipeline(
        augmentation_method=cfg['augmentation'],
        device=DEVICE,
    )
    p.fit(
        train_df['claim'].tolist(), train_df['label'].tolist(),
        val_df['claim'].tolist(), val_df['label'].tolist(),
        epochs=10, retrieve_evidence=cfg['retrieve'], verbose=False,
    )
    preds = p.predict(test_df['claim'].tolist(), retrieve_evidence=cfg['retrieve'])
    acc = accuracy_score(y_test, preds)
    f1 = f1_score(y_test, preds, average='macro')
    print(f"  acc={acc:.4f}  f1_macro={f1:.4f}")
    ablation_results.append({'variant': cfg['name'], 'test_acc': acc, 'test_f1': f1})

df_ablation = pd.DataFrame(ablation_results)
df_ablation.to_csv(RESULTS_DIR / 'ablation_results.csv', index=False)
print('\nAblation results:')
df_ablation

In [ ]:
# Cell 8: Visualizations

import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', font_scale=1.2)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('ARPD Experimental Results', fontsize=15, fontweight='bold')

# --- Plot 1: Model comparison (accuracy & F1) ---
ax = axes[0]
model_data = pd.concat([
    pd.read_csv(RESULTS_DIR / 'baseline_results.csv').rename(columns={'test_f1': 'test_f1_macro'}),
    pd.read_csv(RESULTS_DIR / 'arpd_results.csv').rename(columns={'test_f1': 'test_f1_macro'}),
], ignore_index=True)

x = np.arange(len(model_data))
w = 0.35
bars1 = ax.bar(x - w/2, model_data['test_acc'], w, label='Accuracy', color='steelblue', alpha=0.85)
bars2 = ax.bar(x + w/2, model_data['test_f1_macro'], w, label='F1-Macro', color='coral', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(model_data['model'], rotation=15, ha='right')
ax.set_ylim(0.4, 0.85)
ax.set_ylabel('Score')
ax.set_title('Model Comparison')
ax.legend()
for bar in list(bars1) + list(bars2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

# --- Plot 2: Robustness under paraphrase attack ---
ax = axes[1]
df_rob = pd.read_csv(RESULTS_DIR / 'robustness_results.csv')
ax.plot(df_rob['p_sub'], df_rob['accuracy'], 'o-', color='steelblue', label='Accuracy', linewidth=2)
ax.plot(df_rob['p_sub'], df_rob['f1_macro'], 's--', color='coral', label='F1-Macro', linewidth=2)
ax.set_xlabel('Substitution Rate p')
ax.set_ylabel('Score')
ax.set_title('Robustness Under Synonym Attack')
ax.legend()
ax.set_xlim(-0.02, 0.55)

# --- Plot 3: Training loss curve ---
ax = axes[2]
history_path = RESULTS_DIR / 'arpd_history.csv'
if history_path.exists():
    df_hist = pd.read_csv(history_path)
    ax.plot(df_hist['epoch'], df_hist['loss'], 'o-', color='steelblue', label='Train Loss', linewidth=2)
    ax2 = ax.twinx()
    ax2.plot(df_hist['epoch'], df_hist['f1_macro'], 's--', color='coral', label='Val F1', linewidth=2)
    ax2.set_ylabel('Validation F1-Macro', color='coral')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Training Loss', color='steelblue')
    ax.set_title('Training Curve')
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, loc='center right')
else:
    ax.text(0.5, 0.5, 'Run Cell 5 first', ha='center', va='center', transform=ax.transAxes)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'arpd_results_plot.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved.')

In [ ]:
# Cell 9: Export kết quả tổng hợp ra CSV

import pandas as pd
from pathlib import Path

RESULTS_DIR = Path('/content/ARPD-research/results')

# Tổng hợp tất cả kết quả
summary_rows = []

# Baselines
if (RESULTS_DIR / 'baseline_results.csv').exists():
    df_b = pd.read_csv(RESULTS_DIR / 'baseline_results.csv')
    for _, row in df_b.iterrows():
        summary_rows.append({
            'model': row['model'],
            'category': 'baseline',
            'test_acc': row['test_acc'],
            'test_f1_macro': row.get('test_f1', row.get('test_f1_macro', None)),
        })

# ARPD
if (RESULTS_DIR / 'arpd_results.csv').exists():
    df_a = pd.read_csv(RESULTS_DIR / 'arpd_results.csv')
    for _, row in df_a.iterrows():
        summary_rows.append({
            'model': row['model'],
            'category': 'arpd',
            'test_acc': row['test_acc'],
            'test_f1_macro': row.get('test_f1', row.get('test_f1_macro', None)),
        })

df_summary = pd.DataFrame(summary_rows)
df_summary['test_acc'] = df_summary['test_acc'].map('{:.4f}'.format)
df_summary['test_f1_macro'] = df_summary['test_f1_macro'].map('{:.4f}'.format)

out = RESULTS_DIR / 'ARPD_final_results.csv'
df_summary.to_csv(out, index=False)
print(f'Final results saved → {out}')
print(df_summary.to_string(index=False))

# Download từ Colab
try:
    from google.colab import files
    files.download(str(out))
    print('\nFile downloaded!')
except ImportError:
    print('(Không phải Colab — bỏ qua auto-download)')